# 02 — Bronze (Auto Loader → VARIANT)

Incrementally loads the landed JSON into one `bronze_<entity>` Delta table each, storing the raw payload
as a single `data VARIANT` column (navigate later with `data:field::type`). Auto Loader's checkpoints make
re-runs safe — only new files are read. `availableNow` runs it as a one-shot batch.

In [ ]:
# --- dual-mode setup: runs locally via Databricks Connect AND inside a Databricks Git Folder ---
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
from synergy_schemas import ENTITIES

for e in ENTITIES:
    entity = e["name"]
    source_path     = f"{VOLUME_PATH}/{entity}/"
    checkpoint_path = f"{VOLUME_PATH}/_checkpoints/{entity}"
    target_table    = f"{B}.bronze_{entity}"
    print(f"[{entity}] -> {target_table}")
    try:
        q = (spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.inferColumnTypes", "true")   # infer nested objects as structs (not strings)
             .option("cloudFiles.schemaLocation", checkpoint_path + "/_schema")
             .option("cloudFiles.schemaEvolutionMode", "rescue")
             .option("multiLine", "true")
             .load(source_path)
             # each landed file is a JSON ARRAY of rows; Auto Loader yields one Spark row per element
             .selectExpr("PARSE_JSON(TO_JSON(STRUCT(*))) AS data",
                         "current_timestamp() AS _ingestion_timestamp",
                         "_metadata.file_path AS _source_file")
             .writeStream.option("checkpointLocation", checkpoint_path)
             .trigger(availableNow=True)
             .toTable(target_table))
        q.awaitTermination()
        print(f"  ok: {spark.table(target_table).count():,} rows")
    except Exception as ex:
        print(f"  SKIP/err ({str(ex)[:80]}) — did you run 01_ingest or tests/generate_fake_data first?")

In [ ]:
# peek the raw VARIANT
spark.sql(f"""
  SELECT data:id::string AS id, data:name::string AS team_name,
         data:externalIdMlbam::int AS mlbam, _ingestion_timestamp
  FROM {B}.bronze_teams LIMIT 5
""").show(truncate=False)